## 1.0 Libraries and directories

In [30]:
import ee 
import geemap
import geopandas as gpd
import pandas as pd
import datetime as dt
import pprint as pp
from shapely.geometry import shape

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

roi_name = 'YKF_sub1'
level = 'sr' # 1) 'sr': surface reflectance 2) 'toa' for top of atmosphere
resample_res = 30 
resample_method = 'bilinear'
n_dates = 5 # Number of observation dates
s2_cloud_threshold = 20 # Cloud probability threshold for Sentinel-2
band_dict = {'Sentinel-2': ['B2', #Blue
                            'B3', #Green
                            'B4', #Red
                            'B8'], #NIR
             'LandSat8': ['SR_B2', #Blue
                          'SR_B3', #Green
                          'SR_B4', #Red
                          'SR_B5'] #NIR
}

image_footprints_path = f'./data/overlap_dates_for_roi/{roi_name}_overlap_dates.shp'
best_image_dates = gpd.read_file(image_footprints_path) 
est_utm = f'EPSG:{best_image_dates.estimate_utm_crs().to_epsg()}' # Have to convert pyproj object into literal string for ee
print(f'rois UTM CRS is {est_utm}')


rois UTM CRS is EPSG:32606


## 2.0 Select target dates

In [36]:
best_image_dates['date_plus_1d'] = pd.to_datetime(best_image_dates['date']) + pd.Timedelta(days=1)
best_image_dates = best_image_dates[1:n_dates]

print(best_image_dates[['date', 'per_cover']])

         date  per_cover
1  2021-07-01       81.0
2  2024-07-25       81.0
3  2017-09-08       80.0
4  2022-06-09       73.0


In [34]:
def convert_gpd_geom_to_ee(geom):
        """
        Takes a geopandas geom object and coverts it to an Earth Engine polygon
        """
        coords = list(geom.exterior.coords)
        coords_list = [[x, y] for x, y in coords]
        return ee.Geometry.Polygon(coords_list, proj='EPSG:4326')

def add_1d_to_date(date: str):
        date_plus_1d = pd.to_datetime(date) + pd.Timedelta(days=1)
        date_plus_1d.strftime('%Y-%m-%d')
        return date_plus_1d


def find_s2_col(footprint: gpd.GeoSeries, level: str):

    date = footprint['date']
    date_plus_1d = add_1d_to_date(date)
    polygon = convert_gpd_geom_to_ee(footprint['geometry'])
    
    
    if level == 'sr':
        s2_string = 'COPERNICUS/S2_SR'
    elif level == 'toa':
        print('figure out TOA params lazy dummy')
    else:
        print(f'ERROR: level arg should be "sr" or "toa" not {level}')
    
    s2_col = (
        ee.ImageCollection(s2_string)
        .filterDate(date, date_plus_1d)
        .filterBounds(polygon)
    )

    if s2_col.size().getInfo() == 0:
        print(f'ERROR: No S2 images found for {date} with {level} processing level')
        return None
    
    return s2_col, polygon


def fetch_rescale_s2_imgs(s2_col: ee.ImageCollection,
                          polygon: ee.Geometry,
                          bands: list):
    """
    Generates a single image mosiac with desired bands
    Rescales the bands to match (0-1) surface reflectance range
    TODO: Is rescaling different for TOA??
    """
        
    s2_img = (s2_col.select(bands)
              .mosaic()
              .clip(polygon))
    
    def rescale_s2(img):
        rescaled_bands = img.divide(10_000)
        return rescaled_bands
    
    s2_img = rescale_s2(s2_img)

    return s2_img 

    
def make_s2_cloud_mask(footprint: gpd.GeoSeries, s2_col: ee.ImageCollection, s2_cloud_threshold: int):
    """"""
    date = footprint['date']
    date_plus_1d = add_1d_to_date(date)
    polygon = convert_gpd_geom_to_ee(footprint['geometry'])

    s2_cloud_prob_string = 'COPERNICUS/S2_CLOUD_PROBABILITY'
    s2_clouds = (ee.ImageCollection(s2_cloud_prob_string)
                 .filterBounds(polygon)
                 .filterDate(date, date_plus_1d)
                 .mosaic()
                 .clip(polygon))
    
    s2_scl = (s2_col.select('SCL')
              .mosaic()
              .clip(polygon))
    
    clouds_binary = s2_clouds.select('probability').gt(s2_cloud_threshold).rename('cl_binary')
    s2_shaddow_mask = s2_scl.eq(3)
    s2_cirrus_mask = s2_scl.eq(10) 
    s2_full_mask = clouds_binary.Or(s2_shaddow_mask).Or(s2_cirrus_mask)

    return s2_full_mask

def reduce_mask_resolution(mask: ee.Image, resample_res: int, est_utm: str):
    """Reduces the resolution of cloud masks"""

    print(est_utm)
    original_crs = mask.projection()
    mask_reproj = mask.reproject(
        crs=ee.Projection(est_utm),
        scale=resample_res
    )
    mask_repoj_reduced = mask_reproj.reduceResolution(
        reducer=ee.Reducer.mean()
    ).reproject(
        crs=original_crs,
        scale=resample_res
    )
    return mask_repoj_reduced


def resample_img_and_mask(img: ee.Image, 
                          common_mask: ee.Image, 
                          est_utm: str, 
                          resample_res: int,
                          resample_method: str):
    """
    Resample to match the common cloud mask, then mask the image
    """
    print(est_utm)
    reproj = img.reproject(
         crs=ee.Projection(est_utm),
         scale=resample_res
    )
    resamp = reproj.resample(resample_method).reproject(
         crs=img.projection(),
         scale=resample_res
    )
    
    masked = resamp.updateMask(common_mask.neq(1))
    
    return masked



def s2_processor(footprint: gpd.GeoSeries, 
                level: str, 
                bands: list, 
                resample_res: int,
                resample_method: str,
                est_utm: str, 
                s2_cloud_threshold: int):
    """
    Main function to process and export the Sentinel-2 images
    """
    s2_col, footprint_geom_ee = find_s2_col(footprint, level)
    if s2_col is None:
        return None
        
    s2_img = fetch_rescale_s2_imgs(s2_col, footprint_geom_ee, bands)
    s2_cloud_mask = make_s2_cloud_mask(footprint, s2_col, s2_cloud_threshold)
    s2_cloud_mask = reduce_mask_resolution(s2_cloud_mask, resample_res, est_utm)
    masked_s2 = resample_img_and_mask(s2_img, s2_cloud_mask, est_utm, resample_res, resample_method)
    
    s2_export = ee.batch.Export.image.toDrive(
         image=masked_s2,
         description=f'Sentinel2-{footprint['date']}_{roi_name}',
         fileNamePrefix=f'Sentinel2-{footprint['date']}_{roi_name}',
         folder='scrap',
         scale=30,
         region=footprint_geom_ee,
         crs='EPSG:4326',
         fileFormat='GeoTIFF',
         maxPixels=1e13
    )

    s2_export.start()
    print('Exporting Sentinel-2')
    

def ls_processor(resample_res: int):
    """
    Main function to process and export LandSat-8 images
    """
    return None

def main_processor(some_bullshit):
    return None
    

In [35]:
for idx, row in best_image_dates.iterrows():
    print(row['date'])
    s2_processor(row, 
                level=level, 
                bands=band_dict['Sentinel-2'], 
                resample_res=resample_res,
                resample_method=resample_method,
                est_utm=est_utm,
                s2_cloud_threshold=s2_cloud_threshold
                )

2019-05-16
{'type': 'Polygon',
 'coordinates': [[[-149.25984552746434, 66.2107771736714],
                  [-149.25984552687663, 66.20601610266556],
                  [-149.25975569593592, 66.20601610266556],
                  [-149.2597556957696, 66.20565677660397],
                  [-149.2595760328791, 66.20565677655192],
                  [-149.25957603123268, 66.19972789567672],
                  [-149.25948620135068, 66.19972789567672],
                  [-149.25948620092657, 66.1989194119731],
                  [-149.2593065382939, 66.19891941192103],
                  [-149.2593065364093, 66.192810867989],
                  [-149.25921670676544, 66.192810867989],
                  [-149.259216706322, 66.19191255275699],
                  [-149.25903704370864, 66.19191255270489],
                  [-149.25903704166632, 66.18589384030128],
                  [-149.2589472121802, 66.18589384030128],
                  [-149.25894721172295, 66.18490569354086],
                  [-14

TypeError: cannot unpack non-iterable NoneType object

## 4.0 Clip both image to a common footprint

In [4]:
s2_extent_geom = s2_extent.geometry()
ls_extent_geom = ls_extent.geometry()
img_overlap = ls_extent_geom.intersection(s2_extent_geom, maxError=ee.ErrorMargin(10))

del s2_extent_geom, ls_extent_geom

ls_img = ls_img.clip(img_overlap)
s2_img = s2_img.clip(img_overlap)

# Estimate the utm zone of image overlap
def calculate_utm_zone(polygon):
    centroid = polygon.centroid()
    lon = centroid.coordinates().getNumber(0)
    lat = centroid.coordinates().getNumber(1)
    
    utm_zone = lon.add(180).divide(6).floor().mod(60).add(1)

    is_northern = True
    epsg_code = ee.Number(
        ee.Algorithms.If(
            is_northern,
            utm_zone.add(32600)))
    
    return epsg_code

est_utm = calculate_utm_zone(img_overlap)
est_utm = est_utm.getInfo()
est_utm = f'EPSG:{est_utm}'

## 5.0 Export footprints??

## 6.0 Get a common cloud mask

In [5]:
def get_ls_mask(bounds, date, date_plus1d):
    """Generates a cloud mask for Landsat 8 images using QA_PIXEL bit flags."""
    ls_qa = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
             .filterBounds(bounds)
             .filterDate(date, date_plus1d)
             .select('QA_PIXEL')
             .mosaic()
             .clip(bounds))

    # Define bitmasks for the conditions
    cloudBitMask = 1 << 3        # Bit 3: Cloud
    cloudShadowBitMask = 1 << 4  # Bit 4: Cloud Shadow
    snowBitMask = 1 << 5         # Bit 5: Snow
    cirrusBitMask = 1 << 2       # Bit 2: Cirrus
    dilatedCloudBitMask = 1 << 1 # Bit 1: Dilated Cloud

    # Combine all bitmasks into one
    bitmask = (cloudBitMask
               | cloudShadowBitMask
               | snowBitMask
               | cirrusBitMask
               | dilatedCloudBitMask)

    # Create the mask where any of the bits are set
    ls_full_mask = ls_qa.bitwiseAnd(bitmask).neq(0)

    return ls_full_mask

def get_s2_mask(bounds, date, date_plus1d, resample_proj, target_scale):

    s2_clouds = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
    .filterBounds(bounds)
    .filterDate(date, date_plus1d)
    .mosaic()
    .clip(bounds))

    s2_scl = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .select('SCL')
    .filterBounds(bounds)
    .filterDate(date, date_plus1d)
    .mosaic()
    .clip(bounds))

    clouds_binary = s2_clouds.select('probability').gt(20).rename('cl_binary')
    s2_dark_mask = s2_scl.eq(2)
    s2_shaddow_mask = s2_scl.eq(3)
    s2_cirrus_mask = s2_scl.eq(10) 

    s2_full_mask = clouds_binary.Or(s2_shaddow_mask).Or(s2_cirrus_mask)


    mask_reproj = s2_full_mask.reproject(
        crs=resample_proj,
        scale=target_scale)

    mask_reproj_reduced = mask_reproj.reduceResolution(
        reducer=ee.Reducer.mean()
    ).reproject(
        crs=s2_clouds.projection(), #Back to EPSG:4326
        scale=target_scale)

    return mask_reproj_reduced
    
target_scale=30
ls_mask = get_ls_mask(img_overlap, date, date_plus1d)
s2_mask = get_s2_mask(img_overlap, date, date_plus1d, est_utm, target_scale)

combined_mask = ls_mask.Or(s2_mask)

#### Sieve then dilate the common cloud mask

In [6]:
def sieve_dilate_mask(mask, local_crs, size_threshold, dilation_radius):

    reproj_mask = mask.reproject(
        crs=local_crs,
        scale=30)
    
    connected_pixels = reproj_mask.connectedPixelCount(maxSize=100, eightConnected=True)
    sieved_mask = reproj_mask.updateMask(connected_pixels.gte(size_threshold))
    dilation_kernel = ee.Kernel.circle(radius=dilation_radius, units='meters', normalize=False)
    dilated_mask = sieved_mask.focal_max(kernel=dilation_kernel, iterations=1)

    out_mask = dilated_mask.reproject(
        crs='EPSG:4326',
        scale=30)

    return out_mask

combined_dilated = sieve_dilate_mask(combined_mask, 
                                     local_crs=est_utm, 
                                     size_threshold=50, 
                                     dilation_radius=1000)

## 7.0 Apply Common Mask to Images

In [7]:
def common_mask_s2(img, mask, local_crs, target_scale):
    """ Resample Sentinel-2 to match the common mask (30meters) """
    reproj = img.reproject(
        crs=local_crs,
        scale=target_scale)
    resamp = reproj.resample('bilinear').reproject(
        crs=img.projection(),
        scale=target_scale)

    masked = resamp.updateMask(mask.neq(1))
    
    return masked

s2_masked = common_mask_s2(s2_img, 
                           mask=combined_dilated,
                           local_crs=est_utm, 
                           target_scale=30)

# Landsat8
ls_masked = ls_img.updateMask(combined_dilated.neq(1))



## 8.0 Export the masked images

In [8]:
s2_export = ee.batch.Export.image.toDrive(
    image=s2_masked,
    description=f'Sentinel2_{date}_{roi_name}_resolution{target_scale}',
    fileNamePrefix=f'Sentinel2_{date}_{roi_name}_resolution{target_scale}',
    folder='sentinel2_exports',
    scale=30,
    region=roi,
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e13
)

s2_export.start()

In [9]:
ls_export = ee.batch.Export.image.toDrive(
    image=ls_masked,
    description=f'Landsat8_{date}_{roi_name}_resolution{target_scale}',
    fileNamePrefix=f'Landsat8_{date}_{roi_name}_resolution{target_scale}',
    folder='landsat_exports',
    scale=30,
    region=roi,
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e13
)

In [ ]:
common_mask_export = ee.batch.Export.image.toDrive(
    image=combined_dilated,
    description=f'DilatedCommonMask_{date}_{roi_name}_resolution{target_scale}',
    fileNamePrefix=f'DilatedCommonMask_{date}_{roi_name}_resolution{target_scale}',
    folder='image_common_masks',
    scale=30,
    region=roi,
    crs='EPSG:4326',
    fileFormat='GeoTIFF',
    maxPixels=1e13
)

In [11]:
#s2_export.start()
#ls_export.start()
#common_mask_export.start()